# Bag-of-Words SL

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsSLConfig.get_canonical(
    dataset="homoskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,8133.49707,1098.088257,49548.636719,49984.0,8133.49707,1100.962891,1973.454834,49984.0,2142.287598,1327.385132,49494.257812,49920.0,2142.287598,1284.27832,1999.744507,49920.0
1,4675.945312,1900.248047,49549.277344,49984.0,4675.945312,1635.052246,1973.497192,49984.0,27709.0,3172.587402,49494.257812,49920.0,27709.0,3067.213623,1999.744507,49920.0


In [5]:
analysis = BagOfWordsAnalysisConfig.from_grouped({"example": [(0, config.study_folder)]})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])


In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
0.683594,0.015564,0.761719
0.5546875,-0.263672,-0.055908
0.800781,0.0,-0.238281
0.628906,0.046631,-0.679688
0.165039,-0.279297,0.376953
